# LCD Analysis Dataset Cleaning and Validation

This notebook cleans and validates the LCD-based daily weather dataset for the analysis period (2015–2025).

## Importing Necessities

In [14]:
import pandas as pd
from pathlib import Path

## File paths

In [15]:
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

input_file = DATA_PROCESSED / 'lcd_analysis_2015_2025_daily.csv'
output_file = DATA_PROCESSED / 'lcd_analysis_2015_2025_daily_clean.csv'

## Loading the dataset

In [16]:
df = pd.read_csv(input_file)
print('Shape:', df.shape)
df.head()


Shape: (240739, 12)


,parish,lcd_station_id,lcd_station_name,station_latitude,station_longitude,date,avg_temp,avg_dew_point,avg_relative_humidity,avg_wind_speed,max_temp,min_temp
0,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-01,6.770833,4.251389,84.444444,3.800000,8.6,5.2
1,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-02,13.173611,13.100000,99.472222,2.743056,18.5,8.8
2,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-03,16.840278,16.733333,99.305556,3.327778,20.5,12.0
3,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-04,8.555556,5.820833,83.666667,4.259722,11.4,4.0
4,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2015-01-05,3.455556,-0.893056,74.527778,3.583333,9.0,0.0


## Inspecting column names and data types

In [17]:
print(df.columns.tolist())
print('\nData types before cleaning:')
print(df.dtypes)


['parish', 'lcd_station_id', 'lcd_station_name', 'station_latitude', 'station_longitude', 'date', 'avg_temp', 'avg_dew_point', 'avg_relative_humidity', 'avg_wind_speed', 'max_temp', 'min_temp']

Data types before cleaning:
parish                    object
lcd_station_id            object
lcd_station_name          object
station_latitude         float64
station_longitude        float64
date                      object
avg_temp                 float64
avg_dew_point            float64
avg_relative_humidity    float64
avg_wind_speed           float64
max_temp                 float64
min_temp                 float64
dtype: object


## Converting key columns to proper types

In [18]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

numeric_cols = [
    'station_latitude',
    'station_longitude',
    'avg_temp',
    'avg_dew_point',
    'avg_relative_humidity',
    'avg_wind_speed',
    'max_temp',
    'min_temp'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.dtypes)


parish                           object
lcd_station_id                   object
lcd_station_name                 object
station_latitude                float64
station_longitude               float64
date                     datetime64[ns]
avg_temp                        float64
avg_dew_point                   float64
avg_relative_humidity           float64
avg_wind_speed                  float64
max_temp                        float64
min_temp                        float64
dtype: object


## Missingness summary after type conversion

In [19]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean() * 100
}).sort_values('missing_percent', ascending=False)

missing_summary


,missing_count,missing_percent
avg_relative_humidity,8951,3.718135
avg_dew_point,8950,3.717719
avg_temp,2003,0.832021
max_temp,2003,0.832021
min_temp,2003,0.832021
avg_wind_speed,1929,0.801283
lcd_station_id,0,0.000000
parish,0,0.000000
station_latitude,0,0.000000
lcd_station_name,0,0.000000


## Missingness summary for the key weather variables only

In [20]:
weather_cols = [
    'avg_temp',
    'avg_dew_point',
    'avg_relative_humidity',
    'avg_wind_speed',
    'max_temp',
    'min_temp'
]

weather_missing = pd.DataFrame({
    'missing_count': df[weather_cols].isna().sum(),
    'missing_percent': df[weather_cols].isna().mean() * 100
}).sort_values('missing_percent', ascending=False)

weather_missing

,missing_count,missing_percent
avg_relative_humidity,8951,3.718135
avg_dew_point,8950,3.717719
avg_temp,2003,0.832021
max_temp,2003,0.832021
min_temp,2003,0.832021
avg_wind_speed,1929,0.801283


## Checking for duplicate parish-date rows

In [21]:
dup_count = df.duplicated(subset=['parish', 'date']).sum()
print('Duplicate parish-date rows:', dup_count)

Duplicate parish-date rows: 0


## Inspecting a few rows with missing key weather fields

In [22]:
df[df[weather_cols].isna().any(axis=1)].head(20)


,parish,lcd_station_id,lcd_station_name,station_latitude,station_longitude,date,avg_temp,avg_dew_point,avg_relative_humidity,avg_wind_speed,max_temp,min_temp
882,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-09,NaN,NaN,NaN,1.115385,NaN,NaN
883,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-10,NaN,NaN,NaN,2.760317,NaN,NaN
884,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-11,NaN,NaN,NaN,3.404225,NaN,NaN
885,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-12,NaN,NaN,NaN,3.011429,NaN,NaN
886,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-13,NaN,NaN,NaN,2.598592,NaN,NaN
887,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-14,NaN,NaN,NaN,2.321429,NaN,NaN
888,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-15,NaN,NaN,NaN,2.221538,NaN,NaN
889,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-16,NaN,NaN,NaN,1.938571,NaN,NaN
890,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-17,NaN,NaN,NaN,2.310448,NaN,NaN
891,Acadia,WBAN:00381,"JENNINGS AIRPORT, LA US",30.243,-92.673,2017-06-18,NaN,NaN,NaN,2.453731,NaN,NaN


## Basic range checks for key variables

In [12]:
range_summary = pd.DataFrame({
    'min': df[weather_cols].min(),
    'max': df[weather_cols].max(),
    'mean': df[weather_cols].mean()
})

range_summary

,min,max,mean
avg_temp,-9.9,35.9,20.481900
avg_dew_point,-16.7,29.4,14.973828
avg_relative_humidity,24.0,100.0,73.652953
avg_wind_speed,0.0,19.2,2.659740
max_temp,-7.1,43.3,26.208384
min_temp,-17.1,31.1,14.702668
